# <span style="color:green"> Script for converting CT pics to low quality .nrrd files

## <span style="color:black"> Black: stages in code
## <span style="color:red"> RED = ACTION REQUIRED
## <span style="color:Silver"> Silver = data resulting from action (also can be changed)
## <span style="color:blue"> Blue = notes

## <span style="color:blue"> make sure the CT scans are from the side so that sagital would be top (coronal = frontal)

# Load libraries

In [1]:
from pathlib import Path
import slicer
import os

# Papermill parameters to be changed

In [2]:
# Specify the directory of microCT files (first .jpg file)
microCT_file = "tail4/before ex/CEP_1T/16-tif/CEP_1T_0000.tif"

# if spacing_method = manual → assign the 3 values for spacing (voxel size)
# if spacing_method = automatic → it will be taken from the image data (can double check if spacing in CT scans was saved correctly by comparing it to values in .pca file)

spacing_method = "manual"  # "automatic" or "manual"

spacing = [0.007, 0.007, 0.007] # mmm (Voxelsize taken from .pca file) this is impletmented if spacing_method = manual but if automatic it changes later


# Specify the Region: Specify left, mid, right, full, ...
side = 'full'

# Set the quality resolution 'preview' = 1/4, 'half' = 1/2; 'full' = 1/1 
quality = 'preview'

# Preparing directories

In [3]:
# Specify the parent path
path = Path(microCT_file)
parentpath = path.parent.absolute()
(parentpath / 'segmented_volumes').mkdir(parents=True, exist_ok=True)

# Specify the directory of main folder
folder_path = parentpath / 'segmented_volumes' / side
filepath = str(parentpath) + '/segmented_volumes/' + side

# Prepare 3dSlicer and Import CT data

<span style="color:blue"> For load sequence of .jpg files as a 3D volume into 3D Slicer. 
<span style="color:blue">The [SlicerMoprh](https://github.com/SlicerMorph/SlicerMorph#installation) extension is required!
<span style="color:blue">Steps are reported in 3D Slicer [documentaiton](https://www.slicer.org/wiki/Documentation/4.10/SlicerApplication/MainApplicationGUI#How_to_load_data_from_a_sequence_of_jpg.2C_tif.2C_or_png_files.3F)

Set 'ImageStacks' as currently active module

In [4]:
slicer.util.selectModule('ImageStacks')
# Python scripted modules
moduleWidget = slicer.modules.imagestacks.widgetRepresentation().self()

load image stack

In [5]:
# User selected a scanned image file 
moduleWidget.archetypeText.text = microCT_file

In [6]:
if spacing_method == "automatic":
    spacing_list = moduleWidget.spacingWidget.coordinates
    # Split the string by commas
    spacing = spacing_list.split(',')

    # Convert each element from string to float
    spacing = [float(item) for item in spacing]

In [7]:
# Specify the desired output volume spacing of cylinder
outputVolumeSpacingMm = [spacing[0], spacing[1], spacing[2]] # mm 

# Specify the desired margin value to grow the segments (void and hexa) to touch 
marginValue = spacing[0] # (should be equal or greater than as desired output volume spacing )

<span style="color:blue">The populateFromArchetype method will populate the file list with all files 
that match the numbering pattern in that directory

In [8]:
moduleWidget.populateFromArchetype()

In [9]:
if spacing_method == "manual":
    moduleWidget.spacingWidget.coordinates = f"{spacing[0]},{spacing[1]},{spacing[2]}"

Modifying only the ```logic``` of the ImageStack module does not change the Quality Button at the User Interface (UI). 
However, the new quality is set, as indicated by the output spacing in the UI.

In [10]:
if quality != False:
    moduleWidget.logic.outputQuality = quality
    moduleWidget.updateWidgetFromLogic()

## Instantiate and add a VolumeNode to the scene.

In [11]:
masterVolumeNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLScalarVolumeNode", 'microCT_scan')

Load the files in paths to outputNode.

In [12]:
moduleWidget.logic.loadVolume(outputNode = masterVolumeNode)

Loading failed: User requested cancel
Traceback (most recent call last):
  File "C:/Users/u0151181/AppData/Local/NA-MIC/Slicer 5.0.3/NA-MIC/Extensions-30893/SlicerMorph/lib/Slicer-5.0/qt-scripted-modules/ImageStacks.py", line 415, in onLoadButton
    outputNode = self.logic.loadVolume(self.currentNode(), progressCallback=self.onProgress)
  File "C:/Users/u0151181/AppData/Local/NA-MIC/Slicer 5.0.3/NA-MIC/Extensions-30893/SlicerMorph/lib/Slicer-5.0/qt-scripted-modules/ImageStacks.py", line 671, in loadVolume
    raise ValueError("User requested cancel")
ValueError: User requested cancel


# Save as .nrrd file

In [14]:
# Save a volume node as .nrrd
volume_node = slicer.util.getNode('microCT_scan')
volume_output_file = str(folder_path) + '\microCT_scan_low-quality.nrrd'
slicer.util.saveNode(volume_node, volume_output_file)

True